# Train CNN Models on TrashNet

## Objective 1

Train CNN backbones (ResNet50, EfficientNet-B0 and MobileNetV3-Large) on the TrashNet dataset and save the best-performing model checkpoint for later evaluation.

In [1]:
import os
import json

import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

import config

from dataset import (
get_clean_transforms,
make_trashnet_splits,
set_seed
)

from models import build_model

In [2]:
def evaluate(model, loader, device):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            preds = outputs.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)

            all_labels.extend(labels.numpy())

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return acc, f1

In [3]:
def train_one_backbone(backbone_name):

    set_seed()

    device = config.DEVICE

    print("=" * 60)
    print("Training:", backbone_name)
    print("Device:", device)
    print("=" * 60)

    train_tf, eval_tf = get_clean_transforms()

    train_ds, val_ds, test_ds = make_trashnet_splits(
        train_tf,
        eval_tf
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=2
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    model, _ = build_model(
        backbone_name,
        len(config.TRASHNET_CLASSES),
        pretrained=True
    )

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LR,
        weight_decay=config.WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config.NUM_EPOCHS
    )

    best_val_f1 = -1

    best_ckpt = os.path.join(
        config.CHECKPOINT_DIR,
        f"{backbone_name}_trashnet_best.pt"
    )

    for epoch in range(config.NUM_EPOCHS):

        model.train()

        running_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * images.size(0)

        scheduler.step()

        train_loss = running_loss / len(train_ds)

        val_acc, val_f1 = evaluate(
            model,
            val_loader,
            device
        )

        print(f"Epoch {epoch+1}/{config.NUM_EPOCHS}")
        print(f"Loss={train_loss:.4f}")
        print(f"Validation Accuracy={val_acc:.4f}")
        print(f"Validation Macro F1={val_f1:.4f}")

        if val_f1 > best_val_f1:

            best_val_f1 = val_f1

            torch.save(
                model.state_dict(),
                best_ckpt
            )

    model.load_state_dict(
        torch.load(
            best_ckpt,
            map_location=device
        )
    )

    test_acc, test_f1 = evaluate(
        model,
        test_loader,
        device
    )

    print()
    print("Final Test Accuracy :", test_acc)
    print("Final Test Macro F1 :", test_f1)

    result = {
        "backbone": backbone_name,
        "validation_macro_f1": best_val_f1,
        "trashnet_test_accuracy": test_acc,
        "trashnet_test_macro_f1": test_f1,
        "checkpoint": best_ckpt
    }

    result_path = os.path.join(
        config.RESULTS_DIR,
        f"{backbone_name}_trashnet_results.json"
    )

    with open(result_path, "w") as f:
        json.dump(result, f, indent=2)

    print()
    print("Checkpoint Saved:")
    print(best_ckpt)
    print()
    print("Results Saved:")
    print(result_path)

    return result

In [4]:
BACKBONE = "resnet50"

In [5]:
train_one_backbone(BACKBONE)

Training: resnet50
Device: cpu
Epoch 1/20
Loss=0.8077
Validation Accuracy=0.8417
Validation Macro F1=0.7949
Epoch 2/20
Loss=0.3394
Validation Accuracy=0.8760
Validation Macro F1=0.8409
Epoch 3/20
Loss=0.2595
Validation Accuracy=0.8786
Validation Macro F1=0.8744
Epoch 4/20
Loss=0.1638
Validation Accuracy=0.9024
Validation Macro F1=0.8895
Epoch 5/20
Loss=0.1235
Validation Accuracy=0.9024
Validation Macro F1=0.8912
Epoch 6/20
Loss=0.0647
Validation Accuracy=0.9129
Validation Macro F1=0.8916
Epoch 7/20
Loss=0.0797
Validation Accuracy=0.9103
Validation Macro F1=0.9031
Epoch 8/20
Loss=0.0911
Validation Accuracy=0.9156
Validation Macro F1=0.9100
Epoch 9/20
Loss=0.0240
Validation Accuracy=0.9472
Validation Macro F1=0.9452
Epoch 10/20
Loss=0.0224
Validation Accuracy=0.9367
Validation Macro F1=0.9233
Epoch 11/20
Loss=0.0150
Validation Accuracy=0.9393
Validation Macro F1=0.9289
Epoch 12/20
Loss=0.0185
Validation Accuracy=0.9446
Validation Macro F1=0.9388
Epoch 13/20
Loss=0.0112
Validation Accurac

{'backbone': 'resnet50',
 'validation_macro_f1': 0.9513255235650356,
 'trashnet_test_accuracy': 0.9421052631578948,
 'trashnet_test_macro_f1': 0.9345788597627256,
 'checkpoint': 'e:\\wasteClassificationProject_FIXED\\wasteClassificationProject\\checkpoints\\resnet50_trashnet_best.pt'}

In [6]:
for backbone in config.BACKBONES:

    train_one_backbon
    e(backbone)

Training: resnet50
Device: cpu
Epoch 1/20
Loss=0.8077
Validation Accuracy=0.8417
Validation Macro F1=0.7949
Epoch 2/20
Loss=0.3394
Validation Accuracy=0.8760
Validation Macro F1=0.8409
Epoch 3/20
Loss=0.2595
Validation Accuracy=0.8786
Validation Macro F1=0.8744
Epoch 4/20
Loss=0.1638
Validation Accuracy=0.9024
Validation Macro F1=0.8895
Epoch 5/20
Loss=0.1235
Validation Accuracy=0.9024
Validation Macro F1=0.8912
Epoch 6/20
Loss=0.0647
Validation Accuracy=0.9129
Validation Macro F1=0.8916
Epoch 7/20
Loss=0.0797
Validation Accuracy=0.9103
Validation Macro F1=0.9031
Epoch 8/20
Loss=0.0911
Validation Accuracy=0.9156
Validation Macro F1=0.9100
Epoch 9/20
Loss=0.0240
Validation Accuracy=0.9472
Validation Macro F1=0.9452
Epoch 10/20
Loss=0.0224
Validation Accuracy=0.9367
Validation Macro F1=0.9233
Epoch 11/20
Loss=0.0150
Validation Accuracy=0.9393
Validation Macro F1=0.9289
Epoch 12/20
Loss=0.0185
Validation Accuracy=0.9446
Validation Macro F1=0.9388
Epoch 13/20
Loss=0.0112
Validation Accurac

100.0%


Epoch 1/20
Loss=0.9425
Validation Accuracy=0.8681
Validation Macro F1=0.8282
Epoch 2/20
Loss=0.3464
Validation Accuracy=0.8839
Validation Macro F1=0.8665
Epoch 3/20
Loss=0.1929
Validation Accuracy=0.8892
Validation Macro F1=0.8658
Epoch 4/20
Loss=0.1275
Validation Accuracy=0.9261
Validation Macro F1=0.9119
Epoch 5/20
Loss=0.0787
Validation Accuracy=0.9261
Validation Macro F1=0.9148
Epoch 6/20
Loss=0.0699
Validation Accuracy=0.9182
Validation Macro F1=0.9055
Epoch 7/20
Loss=0.0633
Validation Accuracy=0.8971
Validation Macro F1=0.8956
Epoch 8/20
Loss=0.0541
Validation Accuracy=0.9024
Validation Macro F1=0.8957
Epoch 9/20
Loss=0.0272
Validation Accuracy=0.9314
Validation Macro F1=0.9233
Epoch 10/20
Loss=0.0254
Validation Accuracy=0.9314
Validation Macro F1=0.9294
Epoch 11/20
Loss=0.0312
Validation Accuracy=0.9314
Validation Macro F1=0.9363
Epoch 12/20
Loss=0.0209
Validation Accuracy=0.9367
Validation Macro F1=0.9319
Epoch 13/20
Loss=0.0260
Validation Accuracy=0.9288
Validation Macro F1=0.

100.0%


Epoch 1/20
Loss=0.8181
Validation Accuracy=0.7731
Validation Macro F1=0.7062
Epoch 2/20
Loss=0.2511
Validation Accuracy=0.8865
Validation Macro F1=0.8364
Epoch 3/20
Loss=0.1491
Validation Accuracy=0.8839
Validation Macro F1=0.8491
Epoch 4/20
Loss=0.0820
Validation Accuracy=0.8892
Validation Macro F1=0.8801
Epoch 5/20
Loss=0.0925
Validation Accuracy=0.8945
Validation Macro F1=0.8738
Epoch 6/20
Loss=0.0621
Validation Accuracy=0.9024
Validation Macro F1=0.8778
Epoch 7/20
Loss=0.0300
Validation Accuracy=0.9182
Validation Macro F1=0.9087
Epoch 8/20
Loss=0.0283
Validation Accuracy=0.9182
Validation Macro F1=0.9147
Epoch 9/20
Loss=0.0203
Validation Accuracy=0.9393
Validation Macro F1=0.9348
Epoch 10/20
Loss=0.0123
Validation Accuracy=0.9288
Validation Macro F1=0.9224
Epoch 11/20
Loss=0.0253
Validation Accuracy=0.9288
Validation Macro F1=0.9299
Epoch 12/20
Loss=0.0133
Validation Accuracy=0.9393
Validation Macro F1=0.9373
Epoch 13/20
Loss=0.0192
Validation Accuracy=0.9393
Validation Macro F1=0.